# 多传感器激光 RGB 建图

Super-LIO 使用 3D 雷达与底盘 IMU 输出去畸变扫描；D435 **仅用 RGB** 投影上色，默认关闭深度。RTAB-Map 用 RGB、激光 ICP 与里程计建立和优化关键帧图。本示例只读设备，不发送速度。

按 `docs/mapping.md` 准备依赖和本地 YAML，将 `RSIM_MAPPING_CONFIG` 设为配置路径。`.pose` 使用 graphmap Pose，`.rgb_map` 含 XYZ/RGB 和 `color_valid`，`.map` 额外保留稳定来源编号和体素关系。接收时刻估计并非硬件同步，移动建图前仍需验证时间与外参。


In [ ]:
%load_ext autoreload
%autoreload 3 --print --log

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rsim").is_dir() and (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from examples.mapping.mapping import capture, plot_capture
CONFIG = os.environ.get("RSIM_MAPPING_CONFIG", ROOT / 'configs/mapping.yaml')
if not CONFIG:
    print("请设置 RSIM_MAPPING_CONFIG 后运行采集单元格。")

## 在已有事件循环中运行

provider 在共享子进程中运行，原生 SLAM 节点由生命周期管理。退出 Runtime 会释放本次启动的资源。辅助函数通过 `.pose/.rgb_map/.map` 读取数据，每次在根 `assets/mapping/` 下创建独立数据库，保存 PLY、graphmap 视图及关键帧激光/RGB 档案。


In [ ]:
report = None
if CONFIG:
    report = await capture(CONFIG, seconds=15)
    print({key: report[key] for key in ("points", "pose", "readonly", "mmap", "ply")})

## 查看彩色激光地图

图像通过 `scipykit.mtp_initializer` 输出到根 `assets/`。灰色点表示没有有效 RGB 投影，几何仍由激光提供。PLY 只保存 XYZ/RGB，完整来源关系在 graphmap 档案中。静止实验不能验证移动时的标定精度。


In [ ]:
report

In [ ]:
if report:
    plot_capture(report["output"])
    print(report["status"])

## 给来源点绑定特征并导出 IndexDB

下面在采集结果上附加一个演示向量。特征存储以稳定来源编号为键；体素 IndexDB 是可重新生成的视图。实际模型可用 `source_pixels` 对应的 raw RGB 像素提取特征，原图与标定保存在相邻的 `rtabmap.laser/` 档案中。


In [ ]:
import asyncio
import numpy as np
from rsim import GraphMap

if report:
    view = await asyncio.to_thread(GraphMap.load, report["graphmap"])
    source_id = int(view.snapshot["representative_ids"][0])
    view.set_features("example_embedding", [source_id], [np.array([0.1, 0.2, 0.3])])
    point_key = view.voxel_for(source_id)
    print({"revision": view.revision, "source_id": source_id,
           "point_key": point_key, "contributors": len(view.sources_for(point_key)),
           "feature": view.index_db("example_embedding").get(point_key)})
    await asyncio.to_thread(view.save, Path(report["output"]) / "graphmap-with-features")
    # 可用 graphmap.Environment.load(... / "environment") 单独打开标准视图。


## 在应用中持续跟随图优化

```python
from rsim import Runtime, GraphMap
from rsim.devices import Mapper

mapper = Mapper("mapping")  # 连接另一进程启动的 provider
view = GraphMap()
async with Runtime(mapper):
    frame = await mapper.map.get(timeout=60)
    view.update(frame.data)
    source_id = int(view.source_ids[0])
    view.set_features("my_feature", [source_id], [feature_value])
    while True:  # 放入应用管理的 asyncio task，退出时取消
        frame = await mapper.map.get(after=frame.sequence)
        view.update(frame.data)
        current_key = view.voxel_for(source_id)
        records = view.index_db("my_feature").get(current_key)
```

`source_id` 在同一会话内不随回环变化；它表示一次观测，不是跨帧去重后的物理点编号。`point_key` 会随新位姿改变。体素合并保留所有来源特征，拆分后各自跟随来源。异步取帧仍支持时间戳历史查询；无 ROS 应用只需 RSim、DDS 与 graphmap。
